# 🔬 Microplastic Detection - YOLO Training on Colab GPU
## ⚡ MAXIMUM ACCURACY CONFIGURATION

This notebook trains YOLOv8 for microplastic detection with **highest accuracy settings**.

**Before starting:**
1. Go to `Runtime` → `Change runtime type` → Select **GPU** (T4 or better, A100 recommended)
2. Upload your dataset to Google Drive

**Accuracy Settings Used:**
| Setting | Value | Why |
|---------|-------|-----|
| Model | YOLOv8x | Largest, most accurate |
| Image Size | 1280 | Higher resolution = better small object detection |
| Epochs | 300 | More training = better convergence |
| Optimizer | AdamW | Better generalization |
| Cosine LR | Yes | Smooth learning rate decay |
| Multi-scale | Yes | Robust to size variations |

**Dataset structure expected:**
```
MyDrive/
└── mp-detect/
    └── data/
        └── yolo/
            ├── dataset.yaml
            ├── images/
            │   ├── train/
            │   └── val/
            └── labels/
                ├── train/
                └── val/
```

## 📦 Cell 1: Mount Google Drive & Install Dependencies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install Ultralytics YOLO (latest version)
!pip install ultralytics --upgrade --quiet

# Install additional dependencies for best performance
!pip install albumentations --quiet

print("✅ Setup complete!")

## 🖥️ Cell 2: Verify GPU & Check Memory

**For maximum accuracy, you need:**
- T4: Can run with batch=4-8 at imgsz=1280
- A100: Can run with batch=16 at imgsz=1280

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\n{'='*60}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_memory:.1f} GB")
    
    # Recommend batch size based on GPU
    if 'A100' in gpu_name:
        print(f"\n✅ A100 detected! Use BATCH_SIZE=16, IMAGE_SIZE=1280")
    elif 'V100' in gpu_name:
        print(f"\n✅ V100 detected! Use BATCH_SIZE=8, IMAGE_SIZE=1280")
    elif 'T4' in gpu_name:
        print(f"\n⚠️ T4 detected! Use BATCH_SIZE=4, IMAGE_SIZE=1280 (or BATCH_SIZE=8, IMAGE_SIZE=640)")
    else:
        print(f"\nℹ️ Adjust BATCH_SIZE based on memory errors")
print(f"{'='*60}")

## 📁 Cell 3: Setup Dataset Path

**⚠️ EDIT THIS CELL** to match your Drive folder structure!

In [ ]:
# ⚠️ EDIT THESE PATHS TO MATCH YOUR DRIVE STRUCTURE
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/mp-detect"  # Your project folder
DATASET_PATH = f"{DRIVE_PROJECT_PATH}/data/yolo"          # YOLO dataset folder
OUTPUT_PATH = f"{DRIVE_PROJECT_PATH}/experiments"         # Where to save results

# Verify paths exist
import os

print("Checking paths...")
print(f"  Dataset: {DATASET_PATH} - {'✅ Found' if os.path.exists(DATASET_PATH) else '❌ NOT FOUND'}")
print(f"  Output: {OUTPUT_PATH} - {'✅ Found' if os.path.exists(OUTPUT_PATH) else '📁 Will create'}")

# Create output dir if needed
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Check dataset.yaml
yaml_path = f"{DATASET_PATH}/dataset.yaml"
if os.path.exists(yaml_path):
    print(f"\n📄 dataset.yaml contents:")
    with open(yaml_path) as f:
        print(f.read())
else:
    print(f"\n❌ dataset.yaml not found at {yaml_path}")
    print("Create it in the next cell!")

# Count images
train_dir = f"{DATASET_PATH}/images/train"
val_dir = f"{DATASET_PATH}/images/val"
if os.path.exists(train_dir):
    train_count = len([f for f in os.listdir(train_dir) if f.endswith(('.jpg', '.png'))])
    val_count = len([f for f in os.listdir(val_dir) if f.endswith(('.jpg', '.png'))]) if os.path.exists(val_dir) else 0
    print(f"\n📊 Dataset size: {train_count} train, {val_count} val images")

## 📝 Cell 4: Create/Update dataset.yaml (if needed)

Only run this if you need to create the dataset config file.

In [ ]:
# Skip this cell if you already have dataset.yaml on Drive

yaml_content = f"""
# Microplastic Detection Dataset
path: {DATASET_PATH}
train: images/train
val: images/val

# Classes
names:
  0: fiber
  1: film
  2: fragment
"""

# Uncomment below to write the file
# with open(f"{DATASET_PATH}/dataset.yaml", 'w') as f:
#     f.write(yaml_content)
# print("✅ Created dataset.yaml")

print("ℹ️ Uncomment the lines above to create dataset.yaml")

## 🚀 Cell 5: MAXIMUM ACCURACY TRAINING

### Key Settings for Best Accuracy:

| Parameter | Value | Impact on Accuracy |
|-----------|-------|--------------------|
| `model` | **yolov8x.pt** | Largest model = highest accuracy |
| `imgsz` | **1280** | Detects small microplastics better |
| `epochs` | **300** | More training = better convergence |
| `optimizer` | **AdamW** | Better generalization than SGD |
| `cos_lr` | **True** | Smooth LR decay improves final accuracy |
| `close_mosaic` | **50** | Finetune on clean images at end |
| `label_smoothing` | **0.1** | Prevents overconfident predictions |
| `rect` | **False** | Keep disabled for max accuracy |

⚠️ **Note:** This will take several hours but gives the BEST results!

In [ ]:
from ultralytics import YOLO
import torch
import os

# ============================================================================
# 🎯 MAXIMUM ACCURACY CONFIGURATION
# ============================================================================

# Model - Use the LARGEST for best accuracy
MODEL_SIZE = 'yolov8x.pt'  # x = extra large (most accurate)
# Alternatives: yolov8l.pt (large), yolov8m.pt (medium) if memory issues

# Training duration - More epochs = better accuracy
EPOCHS = 300              # Recommended: 200-500 for best results

# Image size - CRITICAL for small object detection!
IMAGE_SIZE = 1280         # Higher = detects smaller microplastics
# Use 640 if you get out-of-memory errors

# Batch size - Adjust based on GPU memory
# A100: 16, V100: 8, T4: 4 (at imgsz=1280)
BATCH_SIZE = 4            # Reduce if OOM error

# Experiment name
EXPERIMENT_NAME = 'microplastic_yolo_max_accuracy'

# ============================================================================
# Load model
# ============================================================================
model = YOLO(MODEL_SIZE)

print(f"\n{'='*70}")
print(f"🚀 MAXIMUM ACCURACY YOLO TRAINING")
print(f"{'='*70}")
print(f"Model:       {MODEL_SIZE} (LARGEST)")
print(f"Epochs:      {EPOCHS}")
print(f"Image Size:  {IMAGE_SIZE} (HIGH RESOLUTION)")
print(f"Batch Size:  {BATCH_SIZE}")
print(f"Output:      {OUTPUT_PATH}/{EXPERIMENT_NAME}")
print(f"{'='*70}")
print(f"⏱️  Estimated time: {EPOCHS * 2} - {EPOCHS * 5} minutes")
print(f"{'='*70}\n")

# ============================================================================
# 🎯 TRAIN WITH MAXIMUM ACCURACY SETTINGS
# ============================================================================
results = model.train(
    # Dataset
    data=f"{DATASET_PATH}/dataset.yaml",
    
    # Core training params
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMAGE_SIZE,
    
    # Hardware
    device=0,
    workers=2,
    
    # ===== ACCURACY OPTIMIZATION =====
    
    # Optimizer - AdamW is better for generalization
    optimizer='AdamW',
    
    # Learning rate - Lower = more stable, better final accuracy
    lr0=0.001,             # Lower initial LR for stability
    lrf=0.01,              # Final LR = lr0 * lrf
    momentum=0.937,        # SGD momentum (also affects AdamW beta1)
    weight_decay=0.0005,   # L2 regularization
    warmup_epochs=5,       # Gradual LR warmup
    warmup_momentum=0.8,   # Warmup momentum
    warmup_bias_lr=0.1,    # Warmup bias LR
    
    # Cosine LR scheduler - Smooth decay for better convergence
    cos_lr=True,
    
    # Label smoothing - Prevents overconfident predictions
    label_smoothing=0.1,
    
    # Close mosaic augmentation at end for fine-tuning on clean images
    close_mosaic=50,       # Last 50 epochs train on clean images
    
    # Multi-scale training - Robust to size variations
    multi_scale=True,      # ⚠️ Uses more memory
    
    # ===== DATA AUGMENTATION (Aggressive for microplastics) =====
    
    augment=True,
    
    # Color augmentation
    hsv_h=0.02,            # Hue variation
    hsv_s=0.8,             # Saturation variation
    hsv_v=0.5,             # Brightness variation
    
    # Geometric augmentation - CRITICAL for microplastics!
    degrees=180,           # FULL rotation (particles can be any orientation)
    translate=0.2,         # Translation (shifted objects)
    scale=0.9,             # Scale variation (0.1 to 1.9x)
    shear=10,              # Shear angle
    perspective=0.001,     # Perspective distortion
    
    # Flip augmentation
    flipud=0.5,            # Vertical flip
    fliplr=0.5,            # Horizontal flip
    
    # Mosaic & Mixup - Great for small objects
    mosaic=1.0,            # Mosaic probability
    mixup=0.15,            # Mixup probability
    copy_paste=0.3,        # Copy-paste augmentation (good for rare objects)
    
    # ===== LOSS WEIGHTS (tune for your dataset) =====
    
    box=7.5,               # Box loss weight
    cls=0.5,               # Classification loss weight
    dfl=1.5,               # Distribution focal loss weight
    
    # ===== TRAINING BEHAVIOR =====
    
    # Early stopping - Disable for maximum training
    patience=0,            # 0 = No early stopping (train all epochs)
    
    # Mixed precision - Faster training
    amp=True,
    
    # Cache images for faster training
    cache='ram',           # Use 'disk' if RAM is limited
    
    # Keep rectangular = False for max accuracy
    rect=False,
    
    # Saving
    project=OUTPUT_PATH,
    name=EXPERIMENT_NAME,
    exist_ok=True,
    save=True,
    save_period=25,        # Checkpoint every 25 epochs
    
    # Plots & logging
    verbose=True,
    plots=True,
    
    # Single class mode - Enable if all types are just "microplastic"
    # single_cls=False,
    
    # Overlap mask for instance segmentation (if using segment model)
    # overlap_mask=True,
)

print(f"\n{'='*70}")
print("✅ MAXIMUM ACCURACY TRAINING COMPLETE!")
print(f"{'='*70}")

## 📊 Cell 6: Evaluate Model

In [ ]:
from ultralytics import YOLO

# Load best model
best_model_path = f"{OUTPUT_PATH}/{EXPERIMENT_NAME}/weights/best.pt"
model = YOLO(best_model_path)

# Validate with same high-resolution settings
print("🔍 Running validation at full resolution...")
metrics = model.val(
    data=f"{DATASET_PATH}/dataset.yaml",
    imgsz=IMAGE_SIZE,      # Use same image size
    batch=BATCH_SIZE,
    conf=0.001,            # Low conf for full PR curve
    iou=0.6,               # IoU threshold for NMS
    max_det=1000,          # Max detections per image
    plots=True,
)

print(f"\n{'='*70}")
print("📊 FINAL VALIDATION RESULTS")
print(f"{'='*70}")
print(f"mAP50:      {metrics.box.map50:.4f}  (Main detection metric)")
print(f"mAP50-95:   {metrics.box.map:.4f}   (Stricter IoU range)")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")
print(f"{'='*70}")

# Per-class metrics
print("\n📈 Per-Class Performance:")
class_names = ['fiber', 'film', 'fragment']
for i, name in enumerate(class_names):
    if i < len(metrics.box.ap50):
        print(f"  {name:12s}: AP50={metrics.box.ap50[i]:.4f}")

## 🖼️ Cell 7: Test on Sample Images

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
import os

# Load model
model = YOLO(f"{OUTPUT_PATH}/{EXPERIMENT_NAME}/weights/best.pt")

# Get some validation images
val_images_dir = f"{DATASET_PATH}/images/val"
test_images = [os.path.join(val_images_dir, f) for f in os.listdir(val_images_dir)[:6]]

# Run prediction with optimized settings
results = model(
    test_images,
    imgsz=IMAGE_SIZE,
    conf=0.25,
    iou=0.45,
    max_det=500,
)

# Display results
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (ax, result) in enumerate(zip(axes, results)):
    img = result.plot()
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f"Detections: {len(result.boxes)}", fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/{EXPERIMENT_NAME}/sample_predictions.png", dpi=150)
plt.show()

print(f"\n✅ Saved to: {OUTPUT_PATH}/{EXPERIMENT_NAME}/sample_predictions.png")

## 🎯 Cell 8: Test Time Augmentation (TTA) for BEST Predictions

TTA runs inference multiple times with augmentations and averages results.
**Slower but MORE ACCURATE!**

In [ ]:
from ultralytics import YOLO

# Load model
model = YOLO(f"{OUTPUT_PATH}/{EXPERIMENT_NAME}/weights/best.pt")

# Run validation with TTA for maximum accuracy
print("🎯 Running validation with Test Time Augmentation...")
print("⚠️ This is slower but gives better results!\n")

metrics_tta = model.val(
    data=f"{DATASET_PATH}/dataset.yaml",
    imgsz=IMAGE_SIZE,
    augment=True,          # Enable TTA!
    plots=True,
)

print(f"\n{'='*70}")
print("📊 RESULTS WITH TEST TIME AUGMENTATION")
print(f"{'='*70}")
print(f"mAP50 (TTA):     {metrics_tta.box.map50:.4f}")
print(f"mAP50-95 (TTA):  {metrics_tta.box.map:.4f}")
print(f"{'='*70}")

## 💾 Cell 9: Download Model

In [ ]:
print(f"\n{'='*70}")
print("📁 YOUR TRAINED MODEL FILES")
print(f"{'='*70}")
print(f"\nBest model:  {OUTPUT_PATH}/{EXPERIMENT_NAME}/weights/best.pt")
print(f"Last model:  {OUTPUT_PATH}/{EXPERIMENT_NAME}/weights/last.pt")
print(f"\nThese are already on your Google Drive! ✅")
print(f"\n💡 Use this model locally:")
print(f"   model = YOLO('path/to/best.pt')")
print(f"   results = model('image.png', imgsz={IMAGE_SIZE})")

## 🔄 Cell 10: Resume Training (if disconnected)

In [ ]:
# ONLY RUN THIS IF YOU NEED TO RESUME AFTER DISCONNECTION

from ultralytics import YOLO

# Load the last checkpoint
last_model_path = f"{OUTPUT_PATH}/{EXPERIMENT_NAME}/weights/last.pt"

print(f"📂 Resuming from: {last_model_path}")
model = YOLO(last_model_path)

# Resume training
results = model.train(resume=True)

print("✅ Training resumed and completed!")

---
## 📋 Summary of Maximum Accuracy Settings

| Category | Setting | Value | Why |
|----------|---------|-------|-----|
| **Model** | Size | YOLOv8x | Largest = most accurate |
| **Resolution** | imgsz | 1280 | Better small object detection |
| **Training** | epochs | 300 | Full convergence |
| **Training** | patience | 0 | No early stopping |
| **Optimizer** | optimizer | AdamW | Better generalization |
| **LR** | cos_lr | True | Smooth decay |
| **LR** | lr0 | 0.001 | Lower = stable |
| **Regularization** | label_smoothing | 0.1 | Prevents overconfidence |
| **Augmentation** | degrees | 180 | Full rotation |
| **Augmentation** | copy_paste | 0.3 | Helps rare classes |
| **Augmentation** | close_mosaic | 50 | Clean finetune at end |
| **Multi-scale** | multi_scale | True | Size robustness |